# 6. Tabellen voor de hoogwaterstand

Python-versie van het R-script **`production-wl10extended-impr.Rmd`** (KNMI, februari 2026).

Dit notebook maakt de tabellen van de hoogwaterstand (zichtjaar 2035) voor de hoofdstations en de overige
stations. Per station en richtingssector wordt een gewone 2-parameter Weibull-staart opnieuw gefit en de tabel
met voorwaardelijke kansen berekend, eerst voor sectoren van 22,5 graden, daarna voor sectoren van 30 graden.
In dit notebook worden geen toevalsgetallen gebruikt.

**Gebruik:** voer eerst `2_production_wind.ipynb` (richtingsverdeling), `3_statistics_wl.ipynb` en
`4_statistics_wl_others.ipynb` uit (die maken `<station>_stats_multi_meas3.pkl`), controleer daarna de instellingen hieronder en voer alle cellen uit.

**Let op:** dit notebook gebruikt de kansen per richtingssector uit de bestanden die notebook 2 maakt:
`hoekvanh_2035_windrichting_22hdeg_versionFeb2026.csv` en `hoekvanh_2035_windrichting_30deg_versionFeb2026.csv`
(in `output_path`). Het aangeleverde R-script stopt met een foutmelding op de regel die deze verdeling inleest
(`Table22h[2, j]` na `read.csv(skip = 1)`). De tabellen van KNMI zijn gemaakt met de kansen uit
`hoekvanh_winddirs.RData` (`Table22h[2, ]` en `Table30[2, ]`); die verschillen in de sector van 225 graden
(en 240 graden) 0,0001 van de csv-bestanden. In die sectoren kan het laatste cijfer van de tabellen daardoor
afwijken van de tabellen van KNMI.

Uitvoer (dezelfde bestanden als het R-script, in `output_path`):
- `<station>_2035_wl_condprob_12hour_{22hdeg,30deg}_versionFeb2026.csv`
- `<station>_2035_wl_Wbltails_{22hdeg,30deg}_versionFeb2026.csv`
- `<station>_2035_wl_stats_refit_{22hdeg,30deg}_versionFeb2026.pkl`

Volgend notebook: `7_dependence.ipynb`.

## Instellingen
De standaardwaarden van de mappen staan in `kuststatistiek/settings.py`.

In [ ]:
from kuststatistiek import settings

input_path = settings.DATA_PATH  # map met de aangeleverde invoerdata
output_path = (
    settings.OUTPUT_PATH
)  # map met de resultaten van notebooks 2-4; de tabellen komen hier ook

# stations waarvoor je tabellen wilt maken (elke selectie en volgorde is goed)
stations = [
    "hansweert",
    "vlissing",
    "os11",
    "hoekvanh",
    "ijmuiden",
    "denhelde",
    "denoever",
    "westters",
    "harlinge",
    "lauwerso",
    "huibertg",
    "delfzijl",
]

## Voorbereiding

In [ ]:
from kuststatistiek.settings import Paths, ProductionBins, FREQ
from kuststatistiek.production import read_direction_table, weibull_refit_wl
from kuststatistiek.rutils import load_pkl

SLR = 0.003 * (2035 - 2019)  # zeespiegelstijging 2019 -> 2035 [m]

paths = Paths(input_path=input_path, output_path=output_path)
pbins = ProductionBins()  # sectoren van 22,5 en 30 graden voor de tabellen

# windrichtingsverdeling van Hoek van Holland (uitvoer van notebook 2), gebruikt voor alle stations
ptab22 = read_direction_table(
    paths.out("hoekvanh_2035_windrichting_22hdeg_versionFeb2026.csv")
)
ptab30 = read_direction_table(
    paths.out("hoekvanh_2035_windrichting_30deg_versionFeb2026.csv")
)

## Weibull-refit en tabellen (R-chunks `wblfit` en `wblfit30`)

In [ ]:
for dirbins, ptab in [("22h", ptab22), ("30", ptab30)]:
    for station in stations:
        f = paths.out(f"{station}_stats_multi_meas3.pkl")
        if not f.exists():
            print(
                f"{station}: {f.name} niet gevonden, overgeslagen (voer eerst notebook 3 of 4 uit)"
            )
            continue
        weibull_refit_wl(station, load_pkl(f), FREQ, pbins, ptab, dirbins, paths, SLR)
        print(f"{station}: waterstandstabellen {dirbins} graden klaar")